# Anomaly Detection Fundamentals

## Learning Objectives
By the end of this notebook, you will:
- Understand what anomaly detection is and its applications
- Know the different types of anomalies
- Apply statistical methods (Z-score, IQR) for outlier detection
- Use Isolation Forest for anomaly detection
- Apply Local Outlier Factor (LOF)
- Understand One-Class SVM for novelty detection

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)

## 1. What is Anomaly Detection?

**Anomaly Detection** (also called outlier detection) is the identification of data points, observations, or patterns that deviate significantly from the expected behavior.

### Real-World Applications

| Domain | Application |
|--------|-------------|
| **Finance** | Credit card fraud detection |
| **Healthcare** | Disease outbreak detection |
| **Manufacturing** | Defect detection in products |
| **Cybersecurity** | Network intrusion detection |
| **IoT** | Sensor malfunction detection |
| **E-commerce** | Fake review detection |

### Types of Anomalies

1. **Point Anomalies**: A single data point is anomalous (e.g., unusual transaction amount)
2. **Contextual Anomalies**: Anomalous in a specific context (e.g., high temperature in winter)
3. **Collective Anomalies**: A collection of data points is anomalous together (e.g., network attack pattern)

In [ ]:
# Generate sample data with anomalies
np.random.seed(42)

# Normal data (2 clusters)
n_normal = 300
normal_data = np.vstack([
    np.random.randn(n_normal // 2, 2) * 0.5 + [2, 2],
    np.random.randn(n_normal // 2, 2) * 0.5 + [-2, -2]
])

# Anomalies (outliers)
n_anomalies = 20
anomalies = np.random.uniform(low=-6, high=6, size=(n_anomalies, 2))

# Combine
X = np.vstack([normal_data, anomalies])
y_true = np.array([0] * n_normal + [1] * n_anomalies)  # 0 = normal, 1 = anomaly

# Visualize
plt.figure(figsize=(10, 6))
plt.scatter(X[y_true == 0, 0], X[y_true == 0, 1], c='blue', label='Normal', alpha=0.6)
plt.scatter(X[y_true == 1, 0], X[y_true == 1, 1], c='red', marker='x', s=100, label='Anomaly')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Sample Data with Anomalies')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Total samples: {len(X)}")
print(f"Normal samples: {n_normal}")
print(f"Anomalies: {n_anomalies} ({n_anomalies/len(X)*100:.1f}%)")

## 2. Statistical Methods

### 2.1 Z-Score Method

The Z-score measures how many standard deviations a point is from the mean.

$$z = \frac{x - \mu}{\sigma}$$

**Rule of thumb**: Points with |z| > 3 are typically considered anomalies.

In [ ]:
def detect_anomalies_zscore(data, threshold=3):
    """
    Detect anomalies using Z-score method.
    
    Parameters:
    -----------
    data : array-like
        Input data
    threshold : float
        Z-score threshold for anomaly detection
    
    Returns:
    --------
    anomaly_mask : array
        Boolean mask where True = anomaly
    """
    z_scores = np.abs(stats.zscore(data))
    # For multivariate: anomaly if ANY feature exceeds threshold
    anomaly_mask = np.any(z_scores > threshold, axis=1)
    return anomaly_mask

# Apply Z-score method
zscore_anomalies = detect_anomalies_zscore(X, threshold=2.5)

plt.figure(figsize=(10, 6))
plt.scatter(X[~zscore_anomalies, 0], X[~zscore_anomalies, 1], c='blue', label='Normal', alpha=0.6)
plt.scatter(X[zscore_anomalies, 0], X[zscore_anomalies, 1], c='red', marker='x', s=100, label='Detected Anomaly')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Z-Score Anomaly Detection (threshold=2.5)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Anomalies detected: {sum(zscore_anomalies)}")
print(f"True anomalies: {n_anomalies}")

### 2.2 IQR (Interquartile Range) Method

The IQR method is more robust to extreme outliers than Z-score.

$$\text{Lower Bound} = Q1 - 1.5 \times IQR$$
$$\text{Upper Bound} = Q3 + 1.5 \times IQR$$

Points outside these bounds are considered anomalies.

In [ ]:
def detect_anomalies_iqr(data, multiplier=1.5):
    """
    Detect anomalies using IQR method.
    
    Parameters:
    -----------
    data : array-like
        Input data
    multiplier : float
        IQR multiplier (1.5 = outlier, 3 = extreme outlier)
    
    Returns:
    --------
    anomaly_mask : array
        Boolean mask where True = anomaly
    """
    Q1 = np.percentile(data, 25, axis=0)
    Q3 = np.percentile(data, 75, axis=0)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR
    
    # Anomaly if ANY feature is outside bounds
    anomaly_mask = np.any((data < lower_bound) | (data > upper_bound), axis=1)
    return anomaly_mask

# Apply IQR method
iqr_anomalies = detect_anomalies_iqr(X, multiplier=2.0)

plt.figure(figsize=(10, 6))
plt.scatter(X[~iqr_anomalies, 0], X[~iqr_anomalies, 1], c='blue', label='Normal', alpha=0.6)
plt.scatter(X[iqr_anomalies, 0], X[iqr_anomalies, 1], c='red', marker='x', s=100, label='Detected Anomaly')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('IQR Anomaly Detection (multiplier=2.0)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Anomalies detected: {sum(iqr_anomalies)}")

## 3. Isolation Forest

### The Idea

Isolation Forest isolates anomalies using random binary trees. The key insight:

**Anomalies are easier to isolate** (fewer splits needed) because they are rare and different.

### How It Works

1. Randomly select a feature
2. Randomly select a split value between min and max
3. Repeat until point is isolated
4. Anomalies have **shorter path lengths** in trees

### Key Parameters

| Parameter | Description | Typical Values |
|-----------|-------------|----------------|
| `contamination` | Expected proportion of anomalies | 0.01 - 0.1 |
| `n_estimators` | Number of trees | 100 (default) |
| `max_samples` | Samples per tree | 'auto' or 256 |

In [ ]:
# Apply Isolation Forest
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.1,  # Expect 10% anomalies
    random_state=42
)

# Fit and predict (-1 = anomaly, 1 = normal)
iso_predictions = iso_forest.fit_predict(X)
iso_anomalies = iso_predictions == -1

# Get anomaly scores (lower = more anomalous)
anomaly_scores = iso_forest.decision_function(X)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Detection result
axes[0].scatter(X[~iso_anomalies, 0], X[~iso_anomalies, 1], c='blue', label='Normal', alpha=0.6)
axes[0].scatter(X[iso_anomalies, 0], X[iso_anomalies, 1], c='red', marker='x', s=100, label='Anomaly')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
axes[0].set_title('Isolation Forest Detection')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Anomaly score heatmap
scatter = axes[1].scatter(X[:, 0], X[:, 1], c=anomaly_scores, cmap='RdYlBu', alpha=0.7)
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].set_title('Anomaly Scores (Red = More Anomalous)')
plt.colorbar(scatter, ax=axes[1], label='Anomaly Score')

plt.tight_layout()
plt.show()

print(f"Anomalies detected: {sum(iso_anomalies)}")
print(f"True anomalies: {n_anomalies}")

## 4. Local Outlier Factor (LOF)

### The Idea

LOF compares the local density of a point to the local densities of its neighbors.

**Anomalies have substantially lower density** than their neighbors.

### How It Works

1. Find k-nearest neighbors for each point
2. Calculate **local reachability density** (LRD) for each point
3. Compare LRD of a point to LRD of its neighbors
4. LOF > 1 indicates lower density than neighbors (potential anomaly)

### Key Parameters

| Parameter | Description | Typical Values |
|-----------|-------------|----------------|
| `n_neighbors` | Number of neighbors to consider | 20 (default) |
| `contamination` | Expected proportion of anomalies | 0.01 - 0.1 |

In [ ]:
# Apply Local Outlier Factor
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.1,
    novelty=False  # For outlier detection (not novelty detection)
)

# Fit and predict (-1 = anomaly, 1 = normal)
lof_predictions = lof.fit_predict(X)
lof_anomalies = lof_predictions == -1

# Get LOF scores (negative of the actual LOF values)
lof_scores = lof.negative_outlier_factor_

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Detection result
axes[0].scatter(X[~lof_anomalies, 0], X[~lof_anomalies, 1], c='blue', label='Normal', alpha=0.6)
axes[0].scatter(X[lof_anomalies, 0], X[lof_anomalies, 1], c='red', marker='x', s=100, label='Anomaly')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
axes[0].set_title('Local Outlier Factor Detection')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# LOF score heatmap
scatter = axes[1].scatter(X[:, 0], X[:, 1], c=lof_scores, cmap='RdYlBu', alpha=0.7)
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].set_title('LOF Scores (Red = More Anomalous)')
plt.colorbar(scatter, ax=axes[1], label='LOF Score')

plt.tight_layout()
plt.show()

print(f"Anomalies detected: {sum(lof_anomalies)}")

## 5. One-Class SVM

### The Idea

One-Class SVM learns a decision boundary that encompasses the "normal" data.

**Points outside the boundary are anomalies.**

### How It Works

1. Map data to high-dimensional space using kernel
2. Find hyperplane that separates data from origin with maximum margin
3. Points on the "wrong side" of the hyperplane are anomalies

### Key Parameters

| Parameter | Description | Typical Values |
|-----------|-------------|----------------|
| `nu` | Upper bound on fraction of anomalies | 0.01 - 0.1 |
| `kernel` | Kernel type | 'rbf' (default) |
| `gamma` | Kernel coefficient | 'scale' or 'auto' |

In [ ]:
# Apply One-Class SVM
oc_svm = OneClassSVM(
    kernel='rbf',
    nu=0.1,  # Expect ~10% anomalies
    gamma='scale'
)

# Fit and predict (-1 = anomaly, 1 = normal)
svm_predictions = oc_svm.fit_predict(X)
svm_anomalies = svm_predictions == -1

# Visualize
plt.figure(figsize=(10, 6))
plt.scatter(X[~svm_anomalies, 0], X[~svm_anomalies, 1], c='blue', label='Normal', alpha=0.6)
plt.scatter(X[svm_anomalies, 0], X[svm_anomalies, 1], c='red', marker='x', s=100, label='Anomaly')

# Plot decision boundary
xx, yy = np.meshgrid(np.linspace(-7, 7, 100), np.linspace(-7, 7, 100))
Z = oc_svm.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)
plt.contour(xx, yy, Z, levels=[0], linewidths=2, colors='green')
plt.contourf(xx, yy, Z, levels=[Z.min(), 0], alpha=0.1, colors=['red'])

plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('One-Class SVM Detection (Green line = Decision Boundary)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Anomalies detected: {sum(svm_anomalies)}")

## 6. Comparing Methods

Let's compare all the methods on our sample data.

In [ ]:
# Comparison of all methods
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

methods = [
    ('Ground Truth', y_true == 1),
    ('Z-Score', zscore_anomalies),
    ('IQR', iqr_anomalies),
    ('Isolation Forest', iso_anomalies),
    ('Local Outlier Factor', lof_anomalies),
    ('One-Class SVM', svm_anomalies)
]

for ax, (name, anomalies) in zip(axes.flat, methods):
    ax.scatter(X[~anomalies, 0], X[~anomalies, 1], c='blue', label='Normal', alpha=0.6, s=20)
    ax.scatter(X[anomalies, 0], X[anomalies, 1], c='red', marker='x', s=80, label='Anomaly')
    ax.set_title(f'{name}\n(Detected: {sum(anomalies)})')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

plt.suptitle('Comparison of Anomaly Detection Methods', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate accuracy metrics
from sklearn.metrics import precision_score, recall_score, f1_score

methods_results = [
    ('Z-Score', zscore_anomalies),
    ('IQR', iqr_anomalies),
    ('Isolation Forest', iso_anomalies),
    ('LOF', lof_anomalies),
    ('One-Class SVM', svm_anomalies)
]

print("Method Performance Comparison")
print("=" * 60)
print(f"{'Method':<20} {'Precision':>10} {'Recall':>10} {'F1-Score':>10}")
print("-" * 60)

for name, predictions in methods_results:
    precision = precision_score(y_true, predictions)
    recall = recall_score(y_true, predictions)
    f1 = f1_score(y_true, predictions)
    print(f"{name:<20} {precision:>10.3f} {recall:>10.3f} {f1:>10.3f}")

print("=" * 60)
print("\nNote: Precision = correct anomalies / detected anomalies")
print("      Recall = detected true anomalies / total true anomalies")

## 7. When to Use Each Method

| Method | Best For | Limitations |
|--------|----------|-------------|
| **Z-Score** | Simple, univariate data | Assumes normal distribution |
| **IQR** | Robust to extreme outliers | May miss subtle anomalies |
| **Isolation Forest** | General purpose, large datasets | Less effective on low-dimensional data |
| **LOF** | Datasets with varying density | Computationally expensive for large datasets |
| **One-Class SVM** | Small to medium datasets | Slow training, requires tuning |

### Decision Guide

```
Is your data high-dimensional?
├── Yes → Isolation Forest (fast, scalable)
└── No
    ├── Does density vary across data?
    │   ├── Yes → LOF (handles varying density)
    │   └── No → One-Class SVM or Isolation Forest
    └── Simple baseline needed?
        └── Yes → Z-Score or IQR (fast, interpretable)
```

## 8. Practical Example: Transaction Fraud Detection

In [ ]:
# Create synthetic transaction data
np.random.seed(42)

# Normal transactions
n_normal_txn = 1000
normal_amounts = np.random.exponential(50, n_normal_txn)  # Average $50
normal_hour = np.random.randint(8, 22, n_normal_txn)  # 8 AM to 10 PM
normal_frequency = np.random.poisson(3, n_normal_txn)  # ~3 transactions per day

# Fraudulent transactions
n_fraud_txn = 30
fraud_amounts = np.random.uniform(500, 2000, n_fraud_txn)  # Large amounts
fraud_hour = np.random.randint(0, 6, n_fraud_txn)  # Unusual hours
fraud_frequency = np.random.poisson(15, n_fraud_txn)  # Many transactions

# Combine
transactions = pd.DataFrame({
    'amount': np.concatenate([normal_amounts, fraud_amounts]),
    'hour': np.concatenate([normal_hour, fraud_hour]),
    'daily_count': np.concatenate([normal_frequency, fraud_frequency]),
    'is_fraud': [0] * n_normal_txn + [1] * n_fraud_txn
})

print("Transaction Data Sample:")
print(transactions.head(10))
print(f"\nTotal transactions: {len(transactions)}")
print(f"Fraudulent: {transactions['is_fraud'].sum()} ({transactions['is_fraud'].mean()*100:.2f}%)")

In [ ]:
# Prepare features
X_txn = transactions[['amount', 'hour', 'daily_count']].values
y_txn = transactions['is_fraud'].values

# Scale features
scaler = StandardScaler()
X_txn_scaled = scaler.fit_transform(X_txn)

# Apply Isolation Forest
iso_forest_txn = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
predictions = iso_forest_txn.fit_predict(X_txn_scaled)
detected_fraud = predictions == -1

# Evaluate
from sklearn.metrics import classification_report, confusion_matrix

print("Isolation Forest Fraud Detection Results:")
print("=" * 50)
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_txn, detected_fraud))
print(f"\nClassification Report:")
print(classification_report(y_txn, detected_fraud, target_names=['Normal', 'Fraud']))

In [ ]:
# Visualize fraud detection
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Amount vs Hour
axes[0].scatter(transactions.loc[~detected_fraud, 'amount'], 
                transactions.loc[~detected_fraud, 'hour'],
                c='blue', alpha=0.5, label='Normal')
axes[0].scatter(transactions.loc[detected_fraud, 'amount'], 
                transactions.loc[detected_fraud, 'hour'],
                c='red', marker='x', s=100, label='Detected Fraud')
axes[0].set_xlabel('Transaction Amount ($)')
axes[0].set_ylabel('Hour of Day')
axes[0].set_title('Fraud Detection: Amount vs Hour')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Amount vs Daily Count
axes[1].scatter(transactions.loc[~detected_fraud, 'amount'], 
                transactions.loc[~detected_fraud, 'daily_count'],
                c='blue', alpha=0.5, label='Normal')
axes[1].scatter(transactions.loc[detected_fraud, 'amount'], 
                transactions.loc[detected_fraud, 'daily_count'],
                c='red', marker='x', s=100, label='Detected Fraud')
axes[1].set_xlabel('Transaction Amount ($)')
axes[1].set_ylabel('Daily Transaction Count')
axes[1].set_title('Fraud Detection: Amount vs Frequency')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Exercise: Anomaly Detection on Customer Data

**Scenario**: You have customer data and need to identify unusual customers for fraud investigation.

**Tasks:**
1. Load and explore the data
2. Scale the features
3. Apply Isolation Forest with contamination=0.05
4. Apply LOF and compare results
5. Visualize the detected anomalies

In [ ]:
# Create sample customer data
np.random.seed(42)
n_customers = 500

customers = pd.DataFrame({
    'annual_income': np.random.normal(50000, 15000, n_customers),
    'spending_score': np.random.normal(50, 20, n_customers),
    'age': np.random.normal(40, 12, n_customers)
})

# Add some anomalies manually
anomaly_indices = [0, 5, 10, 20, 100]
customers.loc[0, :] = [200000, 100, 25]  # Very high income and spending
customers.loc[5, :] = [10000, 99, 70]    # Low income, very high spending
customers.loc[10, :] = [150000, 5, 30]   # High income, very low spending
customers.loc[20, :] = [45000, 95, 18]   # Young with unusually high spending
customers.loc[100, :] = [180000, 2, 65]  # High income, almost no spending

print("Customer Data Sample:")
print(customers.head(10))

# YOUR CODE HERE
# 1. Scale the features using StandardScaler
# 2. Apply IsolationForest with contamination=0.05
# 3. Apply LocalOutlierFactor with contamination=0.05
# 4. Compare which anomalies each method detects
# 5. Create a scatter plot showing the detected anomalies



## Summary

### Key Takeaways:

1. **Anomaly detection** identifies unusual patterns in data
2. **Statistical methods** (Z-score, IQR) are simple but assume distributions
3. **Isolation Forest** is fast and works well on high-dimensional data
4. **LOF** is good for data with varying densities
5. **One-Class SVM** learns a boundary around normal data
6. **No single method is best** - choose based on your data characteristics

### Quick Reference:

```python
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

# Isolation Forest
iso = IsolationForest(contamination=0.1)
predictions = iso.fit_predict(X)  # -1 = anomaly, 1 = normal

# Local Outlier Factor
lof = LocalOutlierFactor(contamination=0.1)
predictions = lof.fit_predict(X)

# One-Class SVM
svm = OneClassSVM(nu=0.1)
predictions = svm.fit_predict(X)
```

---

**Next:** [../05_Case_Studies/01_Customer_Segmentation.ipynb](../05_Case_Studies/01_Customer_Segmentation.ipynb)